In [6]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import hashlib
import json
import math
import re
from typing import Any

import requests
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

from src.saferesponse_engine import logger
from src.saferesponse_engine.constants import CONFIG_FILE_PATH, PARAM_FILE_PATH, SCHEMA_FILE_PATH
from src.saferesponse_engine.utils.common import create_directories, read_yaml


In [5]:
cd ..

/Users/gojuruakshith/PycharmProjects/SafeResponse_Engine


In [7]:
@dataclass(frozen=True)
class RetrievalConfig:
    root_dir: Path
    query_artifact_path: Path
    faiss_index_path: Path
    retrieval_output_path: Path
    embedding_model: str
    top_k: int
    chunk_size: int
    chunk_overlap: int
    num_articles: int
    min_score_threshold: float


class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAM_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)

    def get_retrieval_layer_config(self) -> RetrievalConfig:
        config = self.config.retrieval_layer
        root_dir = Path(config.root_dir)
        create_directories([root_dir, Path(config.faiss_index_path), Path(config.retrieval_output_path).parent])

        return RetrievalConfig(
            root_dir=root_dir,
            query_artifact_path=Path(config.query_artifact_path),
            faiss_index_path=Path(config.faiss_index_path),
            retrieval_output_path=Path(config.retrieval_output_path),
            embedding_model=str(config.embedding_model),
            top_k=int(config.top_k),
            chunk_size=int(config.chunk_size),
            chunk_overlap=int(config.chunk_overlap),
            num_articles=int(config.num_articles),
            min_score_threshold=float(config.min_score_threshold),
        )


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import load_dataset

class RetrievalLayer:

    def __init__(self, config: RetrievalConfig):
        self.config = config
        logger.info("[Stage 2] Loading BGE-M3 embedding model: %s", config.embedding_model)
        self.embeddings = HuggingFaceEmbeddings(
            model_name=config.embedding_model,          # "BAAI/bge-m3"
            encode_kwargs={"normalize_embeddings": True}
        )
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.chunk_size,
            chunk_overlap=self.config.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""],
        )

    def build_index(self) -> FAISS:
        index_dir     = Path(self.config.faiss_index_path)
        index_marker  = index_dir / "index.faiss"
        metadata_path = index_dir / "index_metadata.json"
        expected_metadata = {
            "corpus":        "wikipedia_20220301_en",
            "embedding":     "bge_m3_1024dim",
            "num_articles":  self.config.num_articles,
        }

        if index_marker.exists() and metadata_path.exists():
            stored_metadata = json.loads(
                metadata_path.read_text(encoding="utf-8")
            )
            if stored_metadata == expected_metadata:
                logger.info(
                    "[Stage 2] Loading existing FAISS index from %s", index_dir
                )
                return FAISS.load_local(
                    str(index_dir),
                    self.embeddings,
                    allow_dangerous_deserialization=True,
                )

        raw_docs = self._load_wikipedia_documents()
        if not raw_docs:
            raise RuntimeError("Unable to build the Wikipedia retrieval corpus.")
        chunks = self.splitter.split_documents(raw_docs)
        for chunk_id, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"]     = chunk_id
            chunk.metadata["char_count"]   = len(chunk.page_content)
            chunk.metadata["content_hash"] = hashlib.sha256(
                chunk.page_content.encode()
            ).hexdigest()[:16]
        logger.info(
            "[Stage 2] Building FAISS vector store with %s chunks", len(chunks)
        )
        vectorstore = FAISS.from_documents(chunks, self.embeddings)
        index_dir.mkdir(parents=True, exist_ok=True)
        vectorstore.save_local(str(index_dir))
        metadata_path.write_text(
            json.dumps(expected_metadata, indent=2), encoding="utf-8"
        )
        logger.info("[Stage 2] FAISS index saved to %s", index_dir)
        return vectorstore

    def retrieve(self) -> list[dict[str, Any]]:
        vectorstore = self.build_index()
        query = self._read_query()
        logger.info("[Stage 2] Query: %s", query)
        results = vectorstore.similarity_search_with_score(
            query, k=self.config.top_k
        )
        retrieved_chunks: list[dict[str, Any]] = []
        for rank, (doc, score) in enumerate(results, start=1):
            retrieved_chunks.append({
                "content":        doc.page_content,
                "source":         doc.metadata.get("source", "unknown"),
                "chunk_id":       doc.metadata.get("chunk_id"),
                "char_count":     doc.metadata.get("char_count"),
                "content_hash":   doc.metadata.get("content_hash"),
                "retrieval_rank": rank,
                "score":          round(float(score), 6),
                "metadata":       doc.metadata,
            })
        scores = [c["score"] for c in retrieved_chunks]
        score_stats = {
            "min":  round(min(scores), 6),
            "max":  round(max(scores), 6),
            "mean": round(sum(scores) / len(scores), 6),
        }
        output = {
            "query":            query,
            "embedding_model":  self.config.embedding_model,
            "top_k":            self.config.top_k,
            "score_stats":      score_stats,
            "chunks":           retrieved_chunks,
        }
        self.config.retrieval_output_path.parent.mkdir(parents=True, exist_ok=True)
        self.config.retrieval_output_path.write_text(
            json.dumps(output, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        logger.info(
            "[Stage 2] Saved %s chunks to %s | scores: %s",
            len(retrieved_chunks),
            self.config.retrieval_output_path,
            score_stats,
        )
        return retrieved_chunks

    def _load_wikipedia_documents(self) -> list[Document]:
        logger.info(
            "[Stage 2] Loading %s Wikipedia articles...", self.config.num_articles
        )
        dataset = load_dataset(
            "wikipedia",
            "20220301.en",
            split=f"train[:{self.config.num_articles}]",
            trust_remote_code=True,
        )
        raw_docs = [
            Document(
                page_content=article["text"],
                metadata={
                    "source":     article["title"],
                    "loader":     "wikipedia_huggingface",
                    "topic_area": "general",
                }
            )
            for article in tqdm(dataset, desc="Converting Wikipedia articles")
        ]
        logger.info("[Stage 2] Loaded %s Wikipedia articles.", len(raw_docs))
        return raw_docs
    def _read_query(self) -> str:
        query = self.config.query_artifact_path.read_text(encoding="utf-8").strip()
        if not query:
            raise ValueError("Query artifact is empty.")
        return query

In [13]:
base_config = ConfigurationManager().get_retrieval_layer_config()

config = RetrievalConfig(
    root_dir               = base_config.root_dir,
    query_artifact_path    = base_config.query_artifact_path,
    faiss_index_path       = base_config.faiss_index_path,
    retrieval_output_path  = base_config.retrieval_output_path,
    embedding_model        = base_config.embedding_model,
    top_k                  = base_config.top_k,
    chunk_size             = base_config.chunk_size,
    chunk_overlap          = base_config.chunk_overlap,
    num_articles           = base_config.num_articles,
    min_score_threshold    = base_config.min_score_threshold,
)

retriever = RetrievalLayer(config)
results = retriever.retrieve()
results[:3]


[2026-04-26 00:39:41,871: INFO: common: YAML file loaded successfully: config/config.yaml]
[2026-04-26 00:39:41,873: INFO: common: YAML file loaded successfully: params.yaml]
[2026-04-26 00:39:41,874: INFO: common: YAML file loaded successfully: schema.yaml]
[2026-04-26 00:39:41,875: INFO: common: Created directory at: artifacts/retrieval]
[2026-04-26 00:39:41,876: INFO: common: Created directory at: artifacts/retrieval/faiss_index]
[2026-04-26 00:39:41,876: INFO: common: Created directory at: artifacts/retrieval]
[2026-04-26 00:39:41,877: INFO: 625074032: [Stage 2] Loading BGE-M3 embedding model: BAAI/bge-m3]
[2026-04-26 00:39:41,900: INFO: model: No device provided, using mps]
[2026-04-26 00:39:42,205: INFO: _client: HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-26 00:39:42,245: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aef

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 45693.15it/s]

[2026-04-26 00:39:43,850: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/main "HTTP/1.1 200 OK"]
[2026-04-26 00:39:43,983: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"]


[2026-04-26 00:39:44,139: INFO: _client: HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"]
[2026-04-26 00:39:44,139: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"]
[2026-04-26 00:39:44,244: INFO: _client: HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-04-26 00:39:44,269: INFO: _client: HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors.index.json "HTTP/1.1 404 Not Found"]
[2026-04-26 00:39:44,343: INFO: _client: HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-04-26 00:39:44,379: INFO: _client: HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors "HTTP/1.1 302 Found"]
[2026-04-26 00:39:44,506: INFO: _client

Downloading: 100%|██████████| 15.3k/15.3k [00:00<00:00, 1.35MB/s]
Downloading: 100%|██████████| 20.3G/20.3G [49:51<00:00, 6.78MB/s]  
Converting Wikipedia articles: 100%|██████████| 15/15 [00:00<00:00, 472.50it/s]

[2026-04-26 01:30:06,969: INFO: 625074032: [Stage 2] Loaded 15 Wikipedia articles.]
[2026-04-26 01:30:07,014: INFO: 625074032: [Stage 2] Building FAISS vector store with 1628 chunks]


[2026-04-26 01:32:03,342: INFO: loader: Loading faiss.]
[2026-04-26 01:32:04,292: INFO: loader: Successfully loaded faiss.]
[2026-04-26 01:32:04,417: INFO: 625074032: [Stage 2] FAISS index saved to artifacts/retrieval/faiss_index]
[2026-04-26 01:32:04,433: INFO: 625074032: [Stage 2] Query: Who invented the telephone and in what year?]
[2026-04-26 01:32:05,283: INFO: 625074032: [Stage 2] Saved 5 chunks to artifacts/retrieval/retrieved_chunks.json | scores: {'min': 1.069143, 'max': 1.148053, 'mean': 1.091995}]


[{'content': 'History',
  'source': 'Autism',
  'chunk_id': 264,
  'char_count': 7,
  'content_hash': '0e76960093379060',
  'retrieval_rank': 1,
  'score': 1.069143,
  'metadata': {'source': 'Autism',
   'loader': 'wikipedia_huggingface',
   'topic_area': 'general',
   'chunk_id': 264,
   'char_count': 7,
   'content_hash': '0e76960093379060'}},
 {'content': 'History',
  'source': 'International Atomic Time',
  'chunk_id': 1478,
  'char_count': 7,
  'content_hash': '0e76960093379060',
  'retrieval_rank': 2,
  'score': 1.069143,
  'metadata': {'source': 'International Atomic Time',
   'loader': 'wikipedia_huggingface',
   'topic_area': 'general',
   'chunk_id': 1478,
   'char_count': 7,
   'content_hash': '0e76960093379060'}},
 {'content': 'Communication',
  'source': 'Autism',
  'chunk_id': 159,
  'char_count': 13,
  'content_hash': '3981a2b9c1ef7fce',
  'retrieval_rank': 3,
  'score': 1.08411,
  'metadata': {'source': 'Autism',
   'loader': 'wikipedia_huggingface',
   'topic_area': 'g